In [ ]:
# ==================== VULNERABILITY COUNT MODELS (LAGGED) ====================
library(fixest)
library(dplyr)
library(tidyr)
library(lubridate)

# ---- 1. Dichotomization functions -------------------------
bin_zero_pos <- function(x) {
  factor(if_else(x < 0.5, "zero", "positive"),
         levels = c("zero", "positive"))
}
bin_ten_less <- function(x) {
  factor(if_else(x > 9.5, "ten", "below_ten"),
         levels = c("below_ten", "ten"))
}
binnings <- list(zero_pos = bin_zero_pos, ten_less = bin_ten_less)

chosen_binning <- c(
  Aggregate_score            = "continuous",
  Maintained_score           = "continuous",
  Code_Review_score          = "continuous",
  Pinned_Dependencies_score  = "zero_pos",
  Security_Policy_score      = "zero_pos",
  Token_Permissions_score    = "zero_pos",
  DependUpTool_score         = "zero_pos",
  Binary_Artifacts_score     = "ten_less",
  Dangerous_Workflow_score   = "ten_less",
  Contrib_score              = "ten_less"
)

controls <- "log1p(downloads_7_day_total) + log1p(dependents) + log1p(loc) + version_age_days + log1p(Release_count)"
fe_part  <- "package_name + year_month"
lag_months <- c(1, 3, 6, 12)

# ---- 2. Function to build a lagged predictor and run one model ------------
run_lagged_model <- function(panel, check_name, binning_type, outcome_var, lag_n) {
  panel_check <- panel %>%
    arrange(package_name, year_month) %>%
    group_by(package_name) %>%
    mutate(lag_raw = dplyr::lag(.data[[check_name]], n = lag_n)) %>%
    ungroup()

  if (binning_type == "continuous") {
    predictor_term <- "lag_raw"
  } else {
    f <- binnings[[binning_type]]
    panel_check$lag_bin <- f(panel_check$lag_raw)
    predictor_term <- "lag_bin"
  }

  fml_str <- paste(
    outcome_var, "~", predictor_term, "+", controls,
    "|", fe_part
  )
  model <- feglm(as.formula(fml_str), data = panel_check,
                  family = "poisson", cluster = ~package_name)
  list(model = model, n_obs = nobs(model), check = check_name,
       outcome = outcome_var, lag = lag_n)
}

# ---- 3. Run for every check x every lag ------------------------------------
panel_vuln <- read.csv("../data/monthly_panel.csv")  # update path as needed

vuln_results <- list()
for (check_name in names(chosen_binning)) {
  vuln_results[[check_name]] <- lapply(lag_months, function(l) {
    run_lagged_model(panel_vuln, check_name, chosen_binning[[check_name]],
                      "Vulnerability_count", l)
  })
  names(vuln_results[[check_name]]) <- paste0("lag", lag_months)
}

# ---- 4. Print summary() for each check x lag model -------------------------
cat("=== Vulnerability Count: Lagged Univariate Model Summaries ===\n\n")
for (check_name in names(vuln_results)) {
  for (lag_label in names(vuln_results[[check_name]])) {
    cat("----", check_name, "|", lag_label, "----\n")
    print(summary(vuln_results[[check_name]][[lag_label]]$model))
    cat("\n")
  }
}

NOTES: 29,636 observations removed because of NA values (RHS: 29,636).
       5,760/0 fixed-effects (19,886 observations) removed because of only 0 outcomes or singletons.

NOTES: 51,965 observations removed because of NA values (RHS: 51,965).
       3,364/0 fixed-effects (13,029 observations) removed because of only 0 outcomes or singletons.

NOTES: 71,929 observations removed because of NA values (RHS: 71,929).
       1,779/0 fixed-effects (7,383 observations) removed because of only 0 outcomes or singletons.

NOTES: 91,951 observations removed because of NA values (RHS: 91,951).
       669/0 fixed-effects (2,422 observations) removed because of only 0 outcomes or singletons.

NOTES: 29,753 observations removed because of NA values (RHS: 29,753).
       5,746/0 fixed-effects (19,769 observations) removed because of only 0 outcomes or singletons.

NOTES: 52,055 observations removed because of NA values (RHS: 52,055).
       3,353/0 fixed-effects (12,939 observations) removed because o

=== Vulnerability Count: Lagged Univariate Model Summaries ===

---- Aggregate_score | lag1 ----
GLM estimation, family = poisson, Dep. Var.: Vulnerability_count
Observations: 54,532
Fixed-effects: package_name: 7,186,  year_month: 24
Standard-errors: Clustered (package_name) 
                              Estimate Std. Error   z value   Pr(>|z|)    
lag_raw                      -0.002036   0.028678 -0.071003 9.4340e-01    
log1p(downloads_7_day_total)  0.002818   0.002365  1.191803 2.3334e-01    
log1p(dependents)             0.010091   0.006060  1.665374 9.5838e-02 .  
log1p(loc)                    0.201221   0.033878  5.939530 2.8584e-09 ***
version_age_days             -0.000577   0.000318 -1.811461 7.0070e-02 .  
log1p(Release_count)         -0.026623   0.008132 -3.273852 1.0609e-03 ** 
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likelihood: -197,941.8   Adj. Pseudo R2: 0.813981
           BIC:  474,574.3     Squared Cor.: 0.886226

---- Aggregate_score 

In [1]:
# ==================== MTTU MODELS (LAGGED) ====================
library(fixest)
library(dplyr)
library(tidyr)
library(lubridate)

# ---- 1. Dichotomization functions -------------------------
bin_zero_pos <- function(x) {
  factor(if_else(x < 0.5, "zero", "positive"),
         levels = c("zero", "positive"))
}
bin_ten_less <- function(x) {
  factor(if_else(x > 9.5, "ten", "below_ten"),
         levels = c("below_ten", "ten"))
}
binnings <- list(zero_pos = bin_zero_pos, ten_less = bin_ten_less)

chosen_binning <- c(
  Aggregate_score            = "continuous",
  Maintained_score           = "continuous",
  Code_Review_score          = "continuous",
  Pinned_Dependencies_score  = "zero_pos",
  Security_Policy_score      = "zero_pos",
  Token_Permissions_score    = "zero_pos",
  DependUpTool_score         = "zero_pos",
  Binary_Artifacts_score     = "ten_less",
  Dangerous_Workflow_score   = "ten_less",
  Contrib_score              = "ten_less"
)

controls <- "log1p(downloads_7_day_total) + log1p(dependents) + log1p(loc) + version_age_days + log1p(Release_count)"
fe_part  <- "package_name + year_month"
lag_months <- c(1, 3, 6, 12)

# ---- 2. Function to build a lagged predictor and run one model ------------
run_lagged_model <- function(panel, check_name, binning_type, outcome_var, lag_n) {
  panel_check <- panel %>%
    arrange(package_name, year_month) %>%
    group_by(package_name) %>%
    mutate(lag_raw = dplyr::lag(.data[[check_name]], n = lag_n)) %>%
    ungroup()

  if (binning_type == "continuous") {
    predictor_term <- "lag_raw"
  } else {
    f <- binnings[[binning_type]]
    panel_check$lag_bin <- f(panel_check$lag_raw)
    predictor_term <- "lag_bin"
  }

  fml_str <- paste(
    outcome_var, "~", predictor_term, "+", controls,
    "|", fe_part
  )
  model <- feglm(as.formula(fml_str), data = panel_check,
                  family = "poisson", cluster = ~package_name)
  list(model = model, n_obs = nobs(model), check = check_name,
       outcome = outcome_var, lag = lag_n)
}

# ---- 3. Run for every check x every lag ------------------------------------
panel_mttu <- read.csv("../data/monthly_panel_mttu_mttr.csv")  

mttu_results <- list()
for (check_name in names(chosen_binning)) {
  mttu_results[[check_name]] <- lapply(lag_months, function(l) {
    run_lagged_model(panel_mttu, check_name, chosen_binning[[check_name]],
                      "MTTU", l)
  })
  names(mttu_results[[check_name]]) <- paste0("lag", lag_months)
}

# ---- 4. Print summary() for each check x lag model -------------------------
cat("=== MTTU: Lagged Univariate Model Summaries ===\n\n")
for (check_name in names(mttu_results)) {
  for (lag_label in names(mttu_results[[check_name]])) {
    cat("----", check_name, "|", lag_label, "----\n")
    print(summary(mttu_results[[check_name]][[lag_label]]$model))
    cat("\n")
  }
}


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘lubridate’


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union


NOTES: 15,212 observations removed because of NA values (RHS: 15,212).
       3,929/0 fixed-effects (12,480 observations) removed because of only 0 outcomes or singletons.

NOTES: 33,979 observations removed because of NA values (RHS: 33,979).
       2,174/0 fixed-effects (7,910 observations) removed because of only 0 outcomes or singletons.

NOTES: 51,234 observations removed because of NA values (RHS: 51,234).
       1,165/0 fixed-effects (4,475 observations) removed because of only 0 outcomes or singletons.

NOTES: 68,764 observations removed because of NA values (RHS: 68,764).
       477/0 fixed-effects (1,702 observations) removed because of only 0 outcomes o

=== MTTU: Lagged Univariate Model Summaries ===

---- Aggregate_score | lag1 ----
GLM estimation, family = poisson, Dep. Var.: MTTU
Observations: 51,371
Fixed-effects: package_name: 6,655,  year_month: 24
Standard-errors: Clustered (package_name) 
                              Estimate Std. Error   z value   Pr(>|z|)    
lag_raw                      -0.112048   0.012111  -9.25174  < 2.2e-16 ***
log1p(downloads_7_day_total) -0.006162   0.003170  -1.94386 5.1913e-02 .  
log1p(dependents)             0.009998   0.007678   1.30206 1.9290e-01    
log1p(loc)                   -0.049858   0.043354  -1.15002 2.5014e-01    
version_age_days              0.003646   0.000495   7.36670 1.7491e-13 ***
log1p(Release_count)         -0.138769   0.013056 -10.62892  < 2.2e-16 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likelihood: -112,703.9   Adj. Pseudo R2: 0.667028
           BIC:  297,908.1     Squared Cor.: 0.877008

---- Aggregate_score | lag3 ----
GLM estimation, fa